# RingWatch Day 8–9
## Explainability, Evidence Gaps, Graph Evidence & Bounded Actions

### Objective

Generate investigation-ready outputs for the Model B top-K investigation queue:

1. SHAP explanations
2. Evidence-gap checklist
3. Graph relationship evidence
4. Deterministic case reports
5. Defense-only action recommendations
6. Investigation audit log

### Safety / Leakage Rules

- No `true_ring_member` for report generation
- No `abuse_ring_id`
- No `ring_type`
- No `population_type`
- No future events
- All evidence is cutoff-aware
- No autonomous account blocking
- No autonomous refund denial
- Actions are recommendations requiring human review where applicable

In [ ]:
# ============================================================
# LOAD CONFIGURATION
# ============================================================

import json
import pickle
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Make the RingWatch project importable when this notebook is run
# from the notebooks directory after a clean kernel restart.
PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "generator").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "generator").exists():
    raise ModuleNotFoundError(
        f"Could not find the RingWatch generator package from {Path.cwd()}"
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from generator.features_config import (
    PREDICTION_CUTOFF,
    FEATURES_GRAPH_PATH,
    PREDICTIONS_TEST_PATH,
    MODEL_B_PATH,
    MODEL_METRICS_PATH,
    EXPLAINABILITY_DIR,
    SHAP_VALUES_PATH,
    SHAP_SUMMARY_PATH,
    EVIDENCE_GAP_PATH,
    GRAPH_EVIDENCE_PATH,
    CASE_REPORTS_PATH,
    BOUNDED_ACTIONS_PATH,
    AUDIT_LOG_PATH,
)

T = pd.Timestamp(PREDICTION_CUTOFF)

print("Project root:", PROJECT_ROOT)
print("Prediction cutoff:", T)
print("Model B:", MODEL_B_PATH)
print("Test predictions:", PREDICTIONS_TEST_PATH)
print("Explainability directory:", EXPLAINABILITY_DIR)


In [62]:
PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "generator").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: d:\CODIN PLAYGROUND\ML-AI\RingWatch


In [ ]:
# Load configuration

from generator.features_config import (
    PREDICTION_CUTOFF,
    FEATURES_GRAPH_PATH,
    PREDICTIONS_TEST_PATH,
    MODEL_B_PATH,
    MODEL_METRICS_PATH,
    EXPLAINABILITY_DIR,
    SHAP_VALUES_PATH,
    SHAP_SUMMARY_PATH,
    EVIDENCE_GAP_PATH,
    GRAPH_EVIDENCE_PATH,
    CASE_REPORTS_PATH,
    BOUNDED_ACTIONS_PATH,
    AUDIT_LOG_PATH,
)

T = pd.Timestamp(PREDICTION_CUTOFF)

print("Prediction cutoff:", T)
print("Model B:", MODEL_B_PATH)
print("Explainability directory:", EXPLAINABILITY_DIR)
print("Test predictions:", PREDICTIONS_TEST_PATH)

Prediction cutoff: 2026-02-20 00:00:00
Model B: D:\CODIN PLAYGROUND\ML-AI\RingWatch\data\processed\model\model_lgbm_B.pkl
Explainability directory: D:\CODIN PLAYGROUND\ML-AI\RingWatch\data\processed\explainability


In [116]:
# Create output directory

EXPLAINABILITY_DIR.mkdir(parents=True, exist_ok=True)

print("Output directory ready:")
print(EXPLAINABILITY_DIR)

Output directory ready:
D:\CODIN PLAYGROUND\ML-AI\RingWatch\data\processed\explainability


In [ ]:
# ============================================================
# LOAD MODEL B
# ============================================================

with open(MODEL_B_PATH, "rb") as f:
    model_B = pickle.load(f)

print("Model B loaded successfully.")
print("Number of trained features:", len(model_B.feature_name_))
print("Features:")
print(model_B.feature_name_)

Model B loaded.
<class 'lightgbm.sklearn.LGBMClassifier'>


In [ ]:
# ============================================================
# LOAD DAY 6–7 TEST PREDICTIONS
# ============================================================

test_predictions = pd.read_csv(
    PREDICTIONS_TEST_PATH
)

test_account_ids = (
    test_predictions["account_id"]
    .dropna()
    .unique()
)

print("Test prediction rows:", len(test_predictions))
print("Unique test accounts:", len(test_account_ids))


# ============================================================
# LOAD GRAPH FEATURES
# ============================================================

features_graph = pd.read_csv(
    FEATURES_GRAPH_PATH
)

test_graph_df = features_graph[
    features_graph["account_id"].isin(test_account_ids)
].copy()


# ============================================================
# VALIDATION
# ============================================================

assert len(test_predictions) == 297, (
    f"Expected 297 test predictions, got {len(test_predictions)}"
)

assert len(test_account_ids) == 297, (
    f"Expected 297 unique test accounts, got {len(test_account_ids)}"
)

assert len(test_graph_df) == 297, (
    f"Expected 297 graph rows, got {len(test_graph_df)}"
)

assert test_graph_df["account_id"].is_unique, (
    "Duplicate account IDs found."
)

assert set(test_graph_df["account_id"]) == set(test_account_ids), (
    "Test graph accounts do not exactly match Day 6–7 test accounts."
)

print("✅ Exact Day 6–7 test set loaded.")
print("✅ No ground truth used.")

Test predictions: 297
Test account IDs: 297
Test graph rows: 297
✅ Day 6–7 test accounts loaded without ground truth


In [ ]:
# ============================================================
# ALIGN TEST FEATURES WITH MODEL B
# ============================================================

model_B_features = list(model_B.feature_name_)

print("Model B feature count:", len(model_B_features))

X_test_B = test_graph_df[
    model_B_features
].copy()


# ============================================================
# VALIDATION
# ============================================================

assert len(model_B_features) == 50, (
    f"Expected 50 Model B features, got {len(model_B_features)}"
)

assert list(X_test_B.columns) == model_B_features, (
    "Test feature order does not match Model B training order."
)

assert X_test_B.shape == (297, 50), (
    f"Unexpected X_test_B shape: {X_test_B.shape}"
)

assert X_test_B.select_dtypes(
    exclude="number"
).shape[1] == 0, (
    "Non-numeric features found."
)

assert np.isfinite(
    X_test_B.to_numpy()
).all(), (
    "NaN or infinite values found."
)

assert "community_id" not in model_B_features, (
    "community_id must not be used."
)

print("✅ Model B feature alignment passed.")
print("X_test_B shape:", X_test_B.shape)


In [120]:
# ============================================================
# LOAD DAY 6–7 OPERATING POINT
# ============================================================

with open(MODEL_METRICS_PATH, "r") as f:
    model_metrics = json.load(f)

K_B = int(model_metrics["model_B"]["operating_k"])

print("Model B operating K:", K_B)

assert K_B > 0

Model B operating K: 7


In [ ]:
# ============================================================
# GENERATE MODEL B SCORES AND RANKING
# ============================================================

proba_B = model_B.predict_proba(
    X_test_B
)[:, 1]

assert len(proba_B) == 297
assert np.isfinite(proba_B).all()


# Rank 1 = highest risk
rank = (
    pd.Series(proba_B)
    .rank(
        ascending=False,
        method="first"
    )
    .astype(int)
    .to_numpy()
)


# Top-K investigation queue
topk_flag = rank <= K_B

flagged_df = test_graph_df.loc[
    topk_flag
].copy()

flagged_df["proba"] = proba_B[topk_flag]
flagged_df["rank"] = rank[topk_flag]
flagged_df["top_k_flag"] = True


# ============================================================
# VALIDATION
# ============================================================

assert topk_flag.sum() == K_B
assert len(flagged_df) == K_B

print("Test accounts:", len(test_graph_df))
print("K_B:", K_B)
print("Flagged accounts:", len(flagged_df))

print("\nInvestigation queue:")

print(
    flagged_df[
        ["account_id", "rank", "proba"]
    ]
    .sort_values("rank")
    .to_string(index=False)
)


In [70]:
# Queue validation

assert len(test_graph_df) == 297
assert len(X_test_B) == 297
assert len(proba_B) == 297
assert topk_flag.sum() == K_B
assert len(flagged_df) == K_B

assert flagged_df["account_id"].is_unique

print("QUEUE VALIDATION PASSED")

QUEUE VALIDATION PASSED


In [71]:
# SHAP calculation

explainer = shap.TreeExplainer(model_B)

shap_values = explainer.shap_values(X_test_B)

if isinstance(shap_values, list):
    shap_values = shap_values[1]

shap_values = np.asarray(shap_values)

print("SHAP shape:", shap_values.shape)
print("Feature matrix shape:", X_test_B.shape)

SHAP shape: (297, 50)
Feature matrix shape: (297, 50)


d:\CODIN PLAYGROUND\ML-AI\RingWatch\.venv\Lib\site-packages\shap\explainers\_tree.py:632: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


In [72]:
# SHAP validation

assert shap_values.shape == X_test_B.shape
assert np.isfinite(shap_values).all()

print("SHAP VALIDATION PASSED")

SHAP VALIDATION PASSED


In [73]:
# ============================================================
# CELL 14 — SHAP DATAFRAME
# ============================================================

# Use the exact feature order used when Model B was trained.
model_B_features = model_B.feature_name_

print("SHAP values shape:", shap_values.shape)
print("Model B feature count:", len(model_B_features))

# Safety check
assert shap_values.shape == (
    len(test_graph_df),
    len(model_B_features)
), (
    f"SHAP shape {shap_values.shape} does not match "
    f"expected {(len(test_graph_df), len(model_B_features))}"
)

# Build SHAP dataframe
shap_df = pd.DataFrame(
    shap_values,
    columns=model_B_features,
    index=test_graph_df.index,
)

# Add account-level metadata
shap_df.insert(
    0,
    "account_id",
    test_graph_df["account_id"].values
)

shap_df.insert(
    1,
    "rank",
    rank.values
)

shap_df.insert(
    2,
    "proba",
    proba_B
)

shap_df.insert(
    3,
    "top_k_flag",
    topk_flag.to_numpy()
)

# Validate
assert shap_df.shape[0] == len(test_graph_df)
assert shap_df.shape[1] == len(model_B_features) + 4
assert np.isfinite(
    shap_df[model_B_features].to_numpy()
).all()

print("\n SHAP DataFrame created successfully")
print("Rows:", len(shap_df))
print("SHAP features:", len(model_B_features))
print("Total columns:", len(shap_df.columns))

shap_df.head()

SHAP values shape: (297, 50)
Model B feature count: 50

 SHAP DataFrame created successfully
Rows: 297
SHAP features: 50
Total columns: 54


,account_id,rank,proba,top_k_flag,total_orders,total_amount,avg_order_value,distinct_devices,distinct_addresses,distinct_phones,...,triangle_count,clustering_coefficient,connected_component_size,shared_edge_count,shared_edge_weight_sum,community_size,community_return_rate,community_refund_rate,community_avg_order_value,community_total_orders
0,A000000,143,2.174334e-07,False,0.000004,-0.003501,-0.010597,9.016052e-14,0.0,4.440892e-16,...,-0.012281,-0.354601,-0.002125,-0.000595,0.857023,-0.000594,-0.075527,-0.006676,0.000977,0.000006
3,A000003,68,4.199160e-07,False,-0.000020,0.040155,-0.016398,-6.198261e-12,0.0,4.440892e-16,...,-0.005593,-0.354487,-0.002125,0.000002,0.999325,-0.000600,-0.071823,-0.005589,0.001276,0.000006
10,A000010,220,1.858040e-07,False,0.000002,-0.000977,0.018716,1.483737e-12,0.0,0.000000e+00,...,0.002885,-0.354112,0.039386,0.000006,-0.911513,0.000632,-0.071597,-0.005526,0.002338,-0.000002
12,A000012,282,1.065973e-07,False,0.000003,-0.000980,0.016204,-2.146853e-11,0.0,4.440892e-16,...,-0.005580,-0.354123,-0.000625,0.000006,-0.869513,-0.000664,-0.071631,-0.005527,0.001085,0.000001
18,A000018,67,4.220351e-07,False,-0.000024,-0.003560,-0.016391,-6.791730e-12,0.0,4.440892e-16,...,-0.012240,-0.354501,-0.002125,-0.000594,1.098501,0.000069,-0.071898,-0.005568,0.000983,-0.000015


In [74]:
# Save SHAP values

shap_df.to_csv(
    SHAP_VALUES_PATH,
    index=False
)

print("Saved:")
print(SHAP_VALUES_PATH)

Saved:
D:\CODIN PLAYGROUND\ML-AI\RingWatch\data\processed\explainability\shap_values_test.csv


In [75]:
# ============================================================
# CELL 15 — SHAP SUMMARY PLOT
# ============================================================

model_B_features = model_B.feature_name_

assert shap_values.shape[1] == len(model_B_features), (
    f"SHAP has {shap_values.shape[1]} features but "
    f"Model B has {len(model_B_features)} features."
)

plt.figure()

shap.summary_plot(
    shap_values,
    X_test_B[model_B_features],
    feature_names=model_B_features,
    show=False,
)

plt.savefig(
    SHAP_SUMMARY_PATH,
    dpi=150,
    bbox_inches="tight"
)

plt.close()

print("✅ SHAP summary plot regenerated.")
print("Features:", len(model_B_features))
print("Saved:", SHAP_SUMMARY_PATH)

✅ SHAP summary plot regenerated.
Features: 50
Saved: D:\CODIN PLAYGROUND\ML-AI\RingWatch\data\processed\explainability\shap_summary.png


In [76]:
# SHAP additivity sanity check

expected_value = explainer.expected_value

if isinstance(expected_value, (list, np.ndarray)):
    expected_value = np.asarray(expected_value).reshape(-1)[-1]

shap_raw_score = (
    shap_values.sum(axis=1)
    + expected_value
)

assert np.isfinite(shap_raw_score).all()

print("SHAP additivity sanity check completed.")
print("Raw-score range:")
print(
    shap_raw_score.min(),
    "→",
    shap_raw_score.max()
)

SHAP additivity sanity check completed.
Raw-score range:
-16.133714440245082 → -4.6415830665300195


In [77]:
# ============================================================
# CELL 18 — TOP SHAP CONTRIBUTORS
# ============================================================

# IMPORTANT:
# Use the exact 50 features used by the trained Model B.
model_B_features = model_B.feature_name_

print("SHAP features:", shap_values.shape[1])
print("Model B features:", len(model_B_features))

assert shap_values.shape[1] == len(model_B_features), (
    "SHAP values and Model B feature names do not match."
)


def get_top_shap_features(
    account_index,
    shap_values,
    feature_names,
    top_n=5
):
    """
    Return the top SHAP contributors for one account.

    SHAP magnitude determines importance.
    Positive SHAP -> pushes prediction toward higher risk.
    Negative SHAP -> pushes prediction toward lower risk.
    """

    values = shap_values[account_index]

    assert len(values) == len(feature_names), (
        f"SHAP value count ({len(values)}) does not match "
        f"feature count ({len(feature_names)})."
    )

    ranking = (
        pd.DataFrame({
            "feature": feature_names,
            "shap_value": values,
        })
        .assign(
            abs_shap=lambda df: df["shap_value"].abs()
        )
        .sort_values(
            "abs_shap",
            ascending=False
        )
        .head(top_n)
        .reset_index(drop=True)
    )

    return ranking


# ------------------------------------------------------------
# Test on first test account
# ------------------------------------------------------------

top_features_example = get_top_shap_features(
    account_index=0,
    shap_values=shap_values,
    feature_names=model_B_features,
    top_n=5
)

print("\nTop SHAP contributors for first test account:")
display(top_features_example)

SHAP features: 50
Model B features: 50

Top SHAP contributors for first test account:


,feature,shap_value,abs_shap
0,shared_edge_weight_sum,0.857023,0.857023
1,account_age_days,-0.416271,0.416271
2,shared_ip_prefix_count,-0.410042,0.410042
3,refund_rate,-0.396429,0.396429
4,clustering_coefficient,-0.354601,0.354601


In [78]:
# Load orders and disputes

orders = pd.read_csv(
    PROJECT_ROOT / "data" / "orders.csv",
    parse_dates=[
        "order_timestamp",
        "delivery_timestamp",
        "return_timestamp",
        "refund_timestamp",
    ],
)

disputes = pd.read_csv(
    PROJECT_ROOT / "data" / "disputes.csv",
    parse_dates=[
        "dispute_created_at"
    ],
)

print("Orders:", len(orders))
print("Disputes:", len(disputes))

print("\nDispute columns:")
print(disputes.columns.tolist())

Orders: 1791
Disputes: 20

Dispute columns:
['dispute_id', 'order_id', 'account_id', 'dispute_created_at', 'dispute_phase', 'dispute_reason_code', 'dispute_reason_category', 'respond_by', 'proof_of_service', 'explanation_letter', 'refund_confirmation', 'access_activity_log', 'refund_cancellation_policy', 'terms_and_conditions']


In [79]:
print(disputes.columns.tolist())

['dispute_id', 'order_id', 'account_id', 'dispute_created_at', 'dispute_phase', 'dispute_reason_code', 'dispute_reason_category', 'respond_by', 'proof_of_service', 'explanation_letter', 'refund_confirmation', 'access_activity_log', 'refund_cancellation_policy', 'terms_and_conditions']


In [80]:
# Validate evidence schema

EVIDENCE_FIELDS = [
    "proof_of_service",
    "explanation_letter",
    "refund_confirmation",
    "access_activity_log",
    "refund_cancellation_policy",
    "terms_and_conditions",
]

missing_fields = [
    col
    for col in EVIDENCE_FIELDS
    if col not in disputes.columns
]

if missing_fields:
    raise ValueError(
        f"Missing evidence columns in disputes.csv: {missing_fields}"
    )

print("Evidence schema validated.")

Evidence schema validated.


In [81]:
# Cutoff filtering

orders_pre = orders[
    orders["order_timestamp"] <= T
].copy()

disputes_pre = disputes[
    disputes["dispute_created_at"] <= T
].copy()

print("Prediction cutoff:", T)
print("Orders before cutoff:", len(orders_pre))
print("Disputes before cutoff:", len(disputes_pre))

Prediction cutoff: 2026-02-20 00:00:00
Orders before cutoff: 1454
Disputes before cutoff: 17


In [82]:
# Evidence helper

def normalize_evidence_value(value):
    """
    Convert evidence field values into boolean where possible.
    """

    if pd.isna(value):
        return False

    if isinstance(value, bool):
        return value

    if isinstance(value, (int, np.integer)):
        return bool(value)

    if isinstance(value, float):
        return bool(value)

    if isinstance(value, str):
        value = value.strip().lower()

        if value in {"true", "yes", "available", "1"}:
            return True

        if value in {"false", "no", "missing", "0"}:
            return False

    return bool(value)

In [83]:
# Build evidence-gap table

evidence_records = []

for account_id in flagged_df["account_id"]:

    account_disputes = disputes_pre[
        disputes_pre["account_id"] == account_id
    ].copy()

    record = {
        "account_id": account_id
    }

    if account_disputes.empty:

        record["has_dispute_at_cutoff"] = False

        for field in EVIDENCE_FIELDS:
            record[field] = "NO_DISPUTE_YET"

        record["missing_evidence_count"] = None

    else:

        record["has_dispute_at_cutoff"] = True

        # Latest dispute for deterministic reporting.
        account_disputes = account_disputes.sort_values(
            "dispute_created_at"
        )

        latest = account_disputes.iloc[-1]

        missing_count = 0

        for field in EVIDENCE_FIELDS:

            available = normalize_evidence_value(
                latest[field]
            )

            record[field] = available

            if not available:
                missing_count += 1

        record["missing_evidence_count"] = missing_count

    evidence_records.append(record)

evidence_df = pd.DataFrame(evidence_records)

evidence_df

,account_id,has_dispute_at_cutoff,proof_of_service,explanation_letter,refund_confirmation,access_activity_log,refund_cancellation_policy,terms_and_conditions,missing_evidence_count
0,A000529,False,NO_DISPUTE_YET,NO_DISPUTE_YET,NO_DISPUTE_YET,NO_DISPUTE_YET,NO_DISPUTE_YET,NO_DISPUTE_YET,None
1,A000842,False,NO_DISPUTE_YET,NO_DISPUTE_YET,NO_DISPUTE_YET,NO_DISPUTE_YET,NO_DISPUTE_YET,NO_DISPUTE_YET,None
2,A000841,False,NO_DISPUTE_YET,NO_DISPUTE_YET,NO_DISPUTE_YET,NO_DISPUTE_YET,NO_DISPUTE_YET,NO_DISPUTE_YET,None
3,A000836,False,NO_DISPUTE_YET,NO_DISPUTE_YET,NO_DISPUTE_YET,NO_DISPUTE_YET,NO_DISPUTE_YET,NO_DISPUTE_YET,None
4,A000838,False,NO_DISPUTE_YET,NO_DISPUTE_YET,NO_DISPUTE_YET,NO_DISPUTE_YET,NO_DISPUTE_YET,NO_DISPUTE_YET,None
5,A000902,False,NO_DISPUTE_YET,NO_DISPUTE_YET,NO_DISPUTE_YET,NO_DISPUTE_YET,NO_DISPUTE_YET,NO_DISPUTE_YET,None
6,A000073,False,NO_DISPUTE_YET,NO_DISPUTE_YET,NO_DISPUTE_YET,NO_DISPUTE_YET,NO_DISPUTE_YET,NO_DISPUTE_YET,None


In [84]:
# Evidence validation

assert len(evidence_df) == len(flagged_df)
assert evidence_df["account_id"].is_unique

for _, row in evidence_df.iterrows():

    if row["has_dispute_at_cutoff"] is False:
        for field in EVIDENCE_FIELDS:
            assert row[field] == "NO_DISPUTE_YET"

    else:
        assert 0 <= row["missing_evidence_count"] <= len(EVIDENCE_FIELDS)

print("EVIDENCE VALIDATION PASSED")

EVIDENCE VALIDATION PASSED


In [85]:
# Save evidence gaps

evidence_df.to_csv(
    EVIDENCE_GAP_PATH,
    index=False
)

print("Saved:")
print(EVIDENCE_GAP_PATH)

Saved:
D:\CODIN PLAYGROUND\ML-AI\RingWatch\data\processed\explainability\evidence_gap_test.csv


In [86]:
# Load graph edges

GRAPH_EDGES_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "account_graph_edges.csv"
)

edges = pd.read_csv(GRAPH_EDGES_PATH)

print("Graph edges:", len(edges))
print(edges.head())

Graph edges: 4299
  account_id_1 account_id_2                  edge_type  weight
0      A000000      A000008               shares_phone     1.0
1      A000000      A000020  shares_payment_instrument     1.0
2      A000000      A000032               shares_phone     1.0
3      A000000      A000034             shares_address     0.7
4      A000000      A000044             shares_address     0.7


In [87]:
# Graph evidence helper

def get_graph_evidence(account_id, edges):

    account_edges = edges[
        (edges["account_id_1"] == account_id)
        |
        (edges["account_id_2"] == account_id)
    ].copy()

    if account_edges.empty:
        return {
            "account_id": account_id,
            "total_graph_links": 0,
            "strongest_edge_type": None,
            "strongest_edge_weight": None,
            "number_of_device_links": 0,
            "number_of_ip_links": 0,
            "number_of_coupon_links": 0,
            "linked_accounts": "",
        }

    linked_accounts = []

    for _, edge in account_edges.iterrows():

        if edge["account_id_1"] == account_id:
            linked = edge["account_id_2"]
        else:
            linked = edge["account_id_1"]

        linked_accounts.append(
            f"{edge['edge_type']} -> {linked}"
        )

    strongest = account_edges.loc[
        account_edges["weight"].idxmax()
    ]

    return {
        "account_id": account_id,
        "total_graph_links": len(account_edges),
        "strongest_edge_type": strongest["edge_type"],
        "strongest_edge_weight": strongest["weight"],
        "number_of_device_links": int(
            (account_edges["edge_type"] == "shares_device").sum()
        ),
        "number_of_ip_links": int(
            (account_edges["edge_type"] == "shares_ip_prefix").sum()
        ),
        "number_of_coupon_links": int(
            (account_edges["edge_type"] == "shares_coupon").sum()
        ),
        "linked_accounts": " | ".join(linked_accounts),
    }

In [88]:
# Generate graph evidence

graph_records = []

for account_id in flagged_df["account_id"]:

    record = get_graph_evidence(
        account_id,
        edges
    )

    graph_records.append(record)

graph_evidence_df = pd.DataFrame(graph_records)

graph_evidence_df

,account_id,total_graph_links,strongest_edge_type,strongest_edge_weight,number_of_device_links,number_of_ip_links,number_of_coupon_links,linked_accounts
0,A000529,14,shares_device,1.0,3,0,0,shares_device -> A000003 | shares_address -> A...
1,A000842,0,NaN,NaN,0,0,0,
2,A000841,0,NaN,NaN,0,0,0,
3,A000836,4,shares_coupon,0.2,0,0,4,shares_coupon -> A000834 | shares_coupon -> A0...
4,A000838,4,shares_coupon,0.2,0,0,4,shares_coupon -> A000834 | shares_coupon -> A0...
5,A000902,11,shares_device,1.0,3,8,0,shares_ip_prefix -> A000800 | shares_ip_prefix...
6,A000073,21,shares_phone,1.0,3,0,0,shares_phone -> A000000 | shares_phone -> A000...


In [89]:
# Graph validation

assert len(graph_evidence_df) == len(flagged_df)
assert graph_evidence_df["account_id"].is_unique

print("GRAPH EVIDENCE VALIDATION PASSED")

GRAPH EVIDENCE VALIDATION PASSED


In [90]:
# Save graph evidence

graph_evidence_df.to_csv(
    GRAPH_EVIDENCE_PATH,
    index=False
)

print("Saved:")
print(GRAPH_EVIDENCE_PATH)

Saved:
D:\CODIN PLAYGROUND\ML-AI\RingWatch\data\processed\explainability\graph_evidence_test.csv


In [91]:
# Prepare case-report data

case_df = flagged_df[
    [
        "account_id",
        "rank",
        "proba",
    ]
].copy()

case_df = case_df.merge(
    evidence_df,
    on="account_id",
    how="left",
    validate="one_to_one"
)

case_df = case_df.merge(
    graph_evidence_df,
    on="account_id",
    how="left",
    validate="one_to_one"
)

case_df.head()

,account_id,rank,proba,has_dispute_at_cutoff,proof_of_service,explanation_letter,refund_confirmation,access_activity_log,refund_cancellation_policy,terms_and_conditions,missing_evidence_count,total_graph_links,strongest_edge_type,strongest_edge_weight,number_of_device_links,number_of_ip_links,number_of_coupon_links,linked_accounts
0,A000529,1,0.009550,False,NO_DISPUTE_YET,NO_DISPUTE_YET,NO_DISPUTE_YET,NO_DISPUTE_YET,NO_DISPUTE_YET,NO_DISPUTE_YET,None,14,shares_device,1.0,3,0,0,shares_device -> A000003 | shares_address -> A...
1,A000842,2,0.000150,False,NO_DISPUTE_YET,NO_DISPUTE_YET,NO_DISPUTE_YET,NO_DISPUTE_YET,NO_DISPUTE_YET,NO_DISPUTE_YET,None,0,NaN,NaN,0,0,0,
2,A000841,3,0.000149,False,NO_DISPUTE_YET,NO_DISPUTE_YET,NO_DISPUTE_YET,NO_DISPUTE_YET,NO_DISPUTE_YET,NO_DISPUTE_YET,None,0,NaN,NaN,0,0,0,
3,A000836,4,0.000136,False,NO_DISPUTE_YET,NO_DISPUTE_YET,NO_DISPUTE_YET,NO_DISPUTE_YET,NO_DISPUTE_YET,NO_DISPUTE_YET,None,4,shares_coupon,0.2,0,0,4,shares_coupon -> A000834 | shares_coupon -> A0...
4,A000838,5,0.000136,False,NO_DISPUTE_YET,NO_DISPUTE_YET,NO_DISPUTE_YET,NO_DISPUTE_YET,NO_DISPUTE_YET,NO_DISPUTE_YET,None,4,shares_coupon,0.2,0,0,4,shares_coupon -> A000834 | shares_coupon -> A0...


In [93]:
# ============================================================
# CELL 32 — CREATE SHAP SUMMARIES FOR FLAGGED ACCOUNTS
# ============================================================

# IMPORTANT:
# Always use the exact feature names from the trained Model B.
model_B_features = model_B.feature_name_

# Safety check
assert shap_values.shape[1] == len(model_B_features), (
    f"SHAP has {shap_values.shape[1]} features, "
    f"but Model B has {len(model_B_features)} features."
)

shap_summaries = {}

for _, row in case_df.iterrows():

    account_id = row["account_id"]

    # Find the account's row in the test dataframe
    matching_indices = test_graph_df.index[
        test_graph_df["account_id"] == account_id
    ]

    assert len(matching_indices) == 1, (
        f"Expected exactly one test row for {account_id}, "
        f"found {len(matching_indices)}."
    )

    idx = matching_indices[0]

    # Convert dataframe index to positional index
    positional_idx = test_graph_df.index.get_loc(idx)

    # Get top SHAP contributors
    top_features = get_top_shap_features(
        account_index=positional_idx,
        shap_values=shap_values,
        feature_names=model_B_features,   # <-- FIX
        top_n=5
    )

    shap_summaries[account_id] = top_features


print("SHAP summaries created for:", len(shap_summaries))

SHAP summaries created for: 7


In [94]:
# Observed behavioral facts helper

BEHAVIORAL_FIELDS = [
    "total_orders",
    "return_rate",
    "refund_rate",
    "dispute_rate",
    "shared_device_count",
    "shared_ip_prefix_count",
    "community_size",
]

available_behavioral_fields = [
    field
    for field in BEHAVIORAL_FIELDS
    if field in test_graph_df.columns
]

print("Behavioral fields used:")
print(available_behavioral_fields)

Behavioral fields used:
['total_orders', 'return_rate', 'refund_rate', 'dispute_rate', 'shared_device_count', 'shared_ip_prefix_count', 'community_size']


In [95]:
# Build case report text

def build_case_report(row, shap_summary):

    account_id = row["account_id"]

    lines = []

    lines.append(
        f"Account ID: {account_id}"
    )

    lines.append(
        f"Risk score (Model B): {row['proba']:.6f}"
    )

    lines.append(
        f"Investigation rank: {int(row['rank'])} / top-{K_B}"
    )

    lines.append("")
    lines.append("Observed facts:")

    account_row = test_graph_df[
        test_graph_df["account_id"] == account_id
    ].iloc[0]

    for field in available_behavioral_fields:

        value = account_row[field]

        if pd.isna(value):
            value = "N/A"

        lines.append(
            f"  {field}: {value}"
        )

    lines.append("")
    lines.append("Top model contributors:")

    for _, shap_row in shap_summary.iterrows():

        direction = (
            "increased"
            if shap_row["shap_value"] > 0
            else "decreased"
        )

        lines.append(
            f"  {shap_row['feature']}: "
            f"{shap_row['shap_value']:.6f} "
            f"({direction} model risk)"
        )

    lines.append("")
    lines.append("Graph evidence:")

    if row["total_graph_links"] == 0:

        lines.append(
            "  No graph relationships observed."
        )

    else:

        linked = str(
            row["linked_accounts"]
        ).split(" | ")

        for relationship in linked:
            lines.append(
                f"  {relationship}"
            )

    lines.append("")
    lines.append("Evidence status:")

    if row["has_dispute_at_cutoff"]:

        for field in EVIDENCE_FIELDS:

            status = (
                "AVAILABLE"
                if row[field] is True
                else "MISSING"
            )

            lines.append(
                f"  {field}: {status}"
            )

        lines.append(
            f"  Missing evidence count: "
            f"{int(row['missing_evidence_count'])}"
        )

    else:

        lines.append(
            "  No dispute observed at prediction cutoff."
        )

    lines.append("")
    lines.append(
        "Recommended action:"
    )

    # Filled after action recommendation.
    lines.append(
        f"  {row['recommended_action']}"
    )

    return "\n".join(lines)

In [96]:
# ============================================================
# RANK-BASED INVESTIGATION PRIORITY
# ============================================================

def get_risk_tier(row):
    """
    Assign an investigation-priority tier based on Model B rank.

    Model B raw probabilities are not calibrated enough for fixed
    probability thresholds. Therefore, the operational queue uses
    the validation-selected top-K ranking.

    This is an investigation priority, NOT a calibrated probability
    of fraud and NOT a ground-truth label.
    """

    rank = int(row["rank"])

    community_size = row.get("community_size", 0)

    if pd.isna(community_size):
        community_size = 0

    # Highest-priority investigations
    if rank <= 2:
        if community_size >= 4:
            return "CRITICAL"
        return "HIGH"

    # Elevated-priority investigations
    if rank <= 5:
        return "MEDIUM"

    # Still flagged, but lower immediate priority
    return "LOW"


def recommend_action(row):
    """
    Map investigation-priority tier to a bounded, defense-only action.

    No automatic blocking, banning, refund denial, or other
    irreversible action is permitted.
    """

    tier = row["risk_tier"]

    if tier == "CRITICAL":
        return (
            "CRITICAL: recommend human review "
            "+ soft-hold refunds"
        )

    if tier == "HIGH":
        return "HIGH: recommend human review"

    if tier == "MEDIUM":
        return (
            "MEDIUM: recommend step-up "
            "verification on refund"
        )

    return "LOW: monitor — no immediate refund action"

In [97]:
# ============================================================
# GENERATE BOUNDED ACTIONS
# ============================================================

actions_df = case_df[
    [
        "account_id",
        "rank",
        "proba",
    ]
].copy()

# Add community size for the CRITICAL-tier condition.
community_lookup = (
    test_graph_df
    .set_index("account_id")["community_size"]
)

actions_df["community_size"] = (
    actions_df["account_id"]
    .map(community_lookup)
)

# Determine investigation priority.
actions_df["risk_tier"] = actions_df.apply(
    get_risk_tier,
    axis=1
)

# Determine bounded defense-only action.
actions_df["recommended_action"] = actions_df.apply(
    recommend_action,
    axis=1
)

# Keep only the intended output columns.
actions_df = actions_df[
    [
        "account_id",
        "rank",
        "proba",
        "risk_tier",
        "recommended_action",
    ]
].sort_values("rank")

print("Investigation actions:")
print(
    actions_df.to_string(index=False)
)

Investigation actions:
account_id  rank    proba risk_tier                                   recommended_action
   A000529     1 0.009550  CRITICAL CRITICAL: recommend human review + soft-hold refunds
   A000842     2 0.000150      HIGH                         HIGH: recommend human review
   A000841     3 0.000149    MEDIUM     MEDIUM: recommend step-up verification on refund
   A000836     4 0.000136    MEDIUM     MEDIUM: recommend step-up verification on refund
   A000838     5 0.000136    MEDIUM     MEDIUM: recommend step-up verification on refund
   A000902     6 0.000120       LOW            LOW: monitor — no immediate refund action
   A000073     7 0.000108       LOW            LOW: monitor — no immediate refund action


In [98]:
# ============================================================
# ACTION VALIDATION
# ============================================================

ALLOWED_TIERS = {
    "CRITICAL",
    "HIGH",
    "MEDIUM",
    "LOW",
}

# Exactly K_B accounts must receive actions.
assert len(actions_df) == K_B, (
    f"Expected {K_B} actions, got {len(actions_df)}"
)

# Every account must be unique.
assert actions_df["account_id"].is_unique, (
    "Duplicate accounts in action output"
)

# Every tier must be allowed.
assert set(
    actions_df["risk_tier"]
).issubset(ALLOWED_TIERS), (
    "Unexpected risk tier detected"
)

# Every flagged account must have an action.
assert set(
    actions_df["account_id"]
) == set(
    flagged_df["account_id"]
), (
    "Action queue does not match investigation queue"
)

# No missing actions.
assert actions_df[
    "recommended_action"
].notna().all()

# No autonomous irreversible actions.
forbidden_actions = [
    "block account",
    "ban account",
    "permanent ban",
    "automatically reject",
]

action_text = (
    actions_df["recommended_action"]
    .str.lower()
)

for forbidden in forbidden_actions:

    assert not action_text.str.contains(
        forbidden,
        regex=False
    ).any(), (
        f"Forbidden action detected: {forbidden}"
    )

# Critical actions must explicitly require human review.
critical_actions = actions_df[
    actions_df["risk_tier"] == "CRITICAL"
]

if len(critical_actions) > 0:

    assert critical_actions[
        "recommended_action"
    ].str.contains(
        "human review",
        case=False,
        regex=False
    ).all()

print("===================================")
print("BOUNDED ACTION VALIDATION PASSED")
print("===================================")

print("\nRisk-tier distribution:")
print(
    actions_df["risk_tier"]
    .value_counts()
)

print("\nAction distribution:")
print(
    actions_df["recommended_action"]
    .value_counts()
)

BOUNDED ACTION VALIDATION PASSED

Risk-tier distribution:
risk_tier
MEDIUM      3
LOW         2
CRITICAL    1
HIGH        1
Name: count, dtype: int64

Action distribution:
recommended_action
MEDIUM: recommend step-up verification on refund        3
LOW: monitor — no immediate refund action               2
CRITICAL: recommend human review + soft-hold refunds    1
HIGH: recommend human review                            1
Name: count, dtype: int64


In [99]:
# Save bounded actions

actions_df.to_csv(
    BOUNDED_ACTIONS_PATH,
    index=False
)

print("Saved:")
print(BOUNDED_ACTIONS_PATH)

Saved:
D:\CODIN PLAYGROUND\ML-AI\RingWatch\data\processed\explainability\bounded_actions_test.csv


In [100]:
# Add action to case dataframe

case_df = case_df.merge(
    actions_df[
        [
            "account_id",
            "risk_tier",
            "recommended_action",
        ]
    ],
    on="account_id",
    how="left",
    validate="one_to_one"
)

print(
    case_df[
        [
            "account_id",
            "rank",
            "proba",
            "risk_tier",
            "recommended_action",
        ]
    ].sort_values("rank")
)

  account_id  rank     proba risk_tier  \
0    A000529     1  0.009550  CRITICAL   
1    A000842     2  0.000150      HIGH   
2    A000841     3  0.000149    MEDIUM   
3    A000836     4  0.000136    MEDIUM   
4    A000838     5  0.000136    MEDIUM   
5    A000902     6  0.000120       LOW   
6    A000073     7  0.000108       LOW   

                                  recommended_action  
0  CRITICAL: recommend human review + soft-hold r...  
1                       HIGH: recommend human review  
2   MEDIUM: recommend step-up verification on refund  
3   MEDIUM: recommend step-up verification on refund  
4   MEDIUM: recommend step-up verification on refund  
5          LOW: monitor — no immediate refund action  
6          LOW: monitor — no immediate refund action  


In [101]:
# Generate case reports

case_reports = []

for _, row in case_df.iterrows():

    account_id = row["account_id"]

    report = build_case_report(
        row,
        shap_summaries[account_id]
    )

    case_reports.append({
        "account_id": account_id,
        "rank": int(row["rank"]),
        "proba": float(row["proba"]),
        "case_report_text": report,
    })

case_reports_df = pd.DataFrame(
    case_reports
)

case_reports_df

,account_id,rank,proba,case_report_text
0,A000529,1,0.009550,Account ID: A000529\nRisk score (Model B): 0.0...
1,A000842,2,0.000150,Account ID: A000842\nRisk score (Model B): 0.0...
2,A000841,3,0.000149,Account ID: A000841\nRisk score (Model B): 0.0...
3,A000836,4,0.000136,Account ID: A000836\nRisk score (Model B): 0.0...
4,A000838,5,0.000136,Account ID: A000838\nRisk score (Model B): 0.0...
5,A000902,6,0.000120,Account ID: A000902\nRisk score (Model B): 0.0...
6,A000073,7,0.000108,Account ID: A000073\nRisk score (Model B): 0.0...


In [ ]:
# ============================================================
# CASE REPORT VALIDATION
# ============================================================

assert len(case_reports_df) == K_B

assert case_reports_df[
    "account_id"
].is_unique

assert case_reports_df[
    "case_report_text"
].notna().all()

# Every case report must contain a bounded action.
assert case_reports_df[
    "case_report_text"
].str.contains(
    "Recommended action:",
    regex=False
).all()

print("===================================")
print("CASE REPORT VALIDATION PASSED")
print("===================================")


In [103]:
# Save case reports

case_reports_df.to_csv(
    CASE_REPORTS_PATH,
    index=False
)

print("Saved:")
print(CASE_REPORTS_PATH)

Saved:
D:\CODIN PLAYGROUND\ML-AI\RingWatch\data\processed\explainability\case_reports_test.csv


In [104]:
# ============================================================
# INVESTIGATION AUDIT LOG
# ============================================================

audit_timestamp = pd.Timestamp.now()

audit_df = actions_df[
    [
        "account_id",
        "proba",
        "rank",
        "risk_tier",
        "recommended_action",
    ]
].copy()

audit_df.insert(
    0,
    "timestamp",
    audit_timestamp
)

audit_df.insert(
    2,
    "model_version",
    "LightGBM_Model_B"
)

audit_df["top_k_flag"] = True

audit_df["action_recommended"] = (
    audit_df["recommended_action"]
)

audit_df["case_report_generated"] = (
    audit_df["account_id"].isin(
        case_reports_df["account_id"]
    )
)

audit_df = audit_df[
    [
        "timestamp",
        "account_id",
        "model_version",
        "proba",
        "rank",
        "risk_tier",
        "top_k_flag",
        "action_recommended",
        "case_report_generated",
    ]
]

print(audit_df.to_string(index=False))

                 timestamp account_id    model_version    proba  rank risk_tier  top_k_flag                                   action_recommended  case_report_generated
2026-08-24 17:48:04.904728    A000529 LightGBM_Model_B 0.009550     1  CRITICAL        True CRITICAL: recommend human review + soft-hold refunds                   True
2026-08-24 17:48:04.904728    A000842 LightGBM_Model_B 0.000150     2      HIGH        True                         HIGH: recommend human review                   True
2026-08-24 17:48:04.904728    A000841 LightGBM_Model_B 0.000149     3    MEDIUM        True     MEDIUM: recommend step-up verification on refund                   True
2026-08-24 17:48:04.904728    A000836 LightGBM_Model_B 0.000136     4    MEDIUM        True     MEDIUM: recommend step-up verification on refund                   True
2026-08-24 17:48:04.904728    A000838 LightGBM_Model_B 0.000136     5    MEDIUM        True     MEDIUM: recommend step-up verification on refund                

In [105]:
# ============================================================
# AUDIT VALIDATION
# ============================================================

assert len(audit_df) == K_B

assert audit_df[
    "account_id"
].is_unique

assert audit_df[
    "case_report_generated"
].all()

assert audit_df[
    "top_k_flag"
].all()

assert audit_df[
    "timestamp"
].notna().all()

assert set(
    audit_df["risk_tier"]
).issubset({
    "CRITICAL",
    "HIGH",
    "MEDIUM",
    "LOW",
})

# Audit accounts must exactly match the investigation queue.
assert set(
    audit_df["account_id"]
) == set(
    flagged_df["account_id"]
)

print("===================================")
print("AUDIT LOG VALIDATION PASSED")
print("===================================")

AUDIT LOG VALIDATION PASSED


In [106]:
# Save audit log

audit_df.to_csv(
    AUDIT_LOG_PATH,
    index=False
)

print("Saved:")
print(AUDIT_LOG_PATH)

Saved:
D:\CODIN PLAYGROUND\ML-AI\RingWatch\data\processed\explainability\investigation_audit_log.csv


In [107]:
# Final Day 8–9 validation

expected_files = [
    SHAP_VALUES_PATH,
    SHAP_SUMMARY_PATH,
    EVIDENCE_GAP_PATH,
    GRAPH_EVIDENCE_PATH,
    CASE_REPORTS_PATH,
    BOUNDED_ACTIONS_PATH,
    AUDIT_LOG_PATH,
]

print("Checking output files...\n")

all_exist = True

for path in expected_files:

    exists = Path(path).exists()

    print(
        f"{'✓' if exists else '✗'} {path}"
    )

    if not exists:
        all_exist = False

assert all_exist

print("\nAll expected artifacts exist.")

Checking output files...

✓ D:\CODIN PLAYGROUND\ML-AI\RingWatch\data\processed\explainability\shap_values_test.csv
✓ D:\CODIN PLAYGROUND\ML-AI\RingWatch\data\processed\explainability\shap_summary.png
✓ D:\CODIN PLAYGROUND\ML-AI\RingWatch\data\processed\explainability\evidence_gap_test.csv
✓ D:\CODIN PLAYGROUND\ML-AI\RingWatch\data\processed\explainability\graph_evidence_test.csv
✓ D:\CODIN PLAYGROUND\ML-AI\RingWatch\data\processed\explainability\case_reports_test.csv
✓ D:\CODIN PLAYGROUND\ML-AI\RingWatch\data\processed\explainability\bounded_actions_test.csv
✓ D:\CODIN PLAYGROUND\ML-AI\RingWatch\data\processed\explainability\investigation_audit_log.csv

All expected artifacts exist.


In [108]:
# Final integrity checks

assert len(flagged_df) == K_B
assert len(evidence_df) == K_B
assert len(graph_evidence_df) == K_B
assert len(case_reports_df) == K_B
assert len(actions_df) == K_B
assert len(audit_df) == K_B

assert case_reports_df[
    "account_id"
].is_unique

assert evidence_df[
    "account_id"
].is_unique

assert graph_evidence_df[
    "account_id"
].is_unique

assert actions_df[
    "account_id"
].is_unique

print("===================================")
print("RINGWATCH DAY 8–9 VALIDATION")
print("===================================")

print(f"Investigation queue: {K_B}")
print(f"SHAP explanations:   {len(shap_df)}")
print(f"Evidence reports:    {len(evidence_df)}")
print(f"Graph evidence:      {len(graph_evidence_df)}")
print(f"Case reports:        {len(case_reports_df)}")
print(f"Actions:             {len(actions_df)}")
print(f"Audit records:       {len(audit_df)}")

print("\nDAY 8–9 PASSED")

RINGWATCH DAY 8–9 VALIDATION
Investigation queue: 7
SHAP explanations:   297
Evidence reports:    7
Graph evidence:      7
Case reports:        7
Actions:             7
Audit records:       7

DAY 8–9 PASSED


In [110]:
# Investigation queue

display(
    case_df[
        [
            "account_id",
            "rank",
            "proba",
            "risk_tier",
            "recommended_action",
            "total_graph_links",
            "missing_evidence_count",
        ]
    ].sort_values("rank")
)

,account_id,rank,proba,risk_tier,recommended_action,total_graph_links,missing_evidence_count
0,A000529,1,0.009550,CRITICAL,CRITICAL: recommend human review + soft-hold r...,14,None
1,A000842,2,0.000150,HIGH,HIGH: recommend human review,0,None
2,A000841,3,0.000149,MEDIUM,MEDIUM: recommend step-up verification on refund,0,None
3,A000836,4,0.000136,MEDIUM,MEDIUM: recommend step-up verification on refund,4,None
4,A000838,5,0.000136,MEDIUM,MEDIUM: recommend step-up verification on refund,4,None
5,A000902,6,0.000120,LOW,LOW: monitor — no immediate refund action,11,None
6,A000073,7,0.000108,LOW,LOW: monitor — no immediate refund action,21,None
